# DSNML Retail Sales Forecasting

This notebook loads the retail sales dataset, performs deterministic feature engineering, and trains a CatBoost regressor for `total_sales` prediction.

## Workflow
1. Load `train.csv` and `test.csv`
2. Clean and engineer features
3. Train CatBoost with 5-fold cross-validation
4. Save `final_catboost_submission.csv`


In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn catboost

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

# Optional: use Google Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/DSNML')


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print('Train shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())

In [ ]:
def prepare_data(df):
    df = df.copy()
    for col in ['product_category', 'fat_content']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').astype(str).str.lower().str.strip()
    if 'product_weight_kg' in df.columns:
        df['product_weight_kg'] = df['product_weight_kg'].fillna(-1)
        df['weight_clean'] = df['product_weight_kg'].replace(-1, np.nan)
        if 'product_price' in df.columns:
            df['price_per_kg'] = df['product_price'] / df['weight_clean']
    if 'store_size' in df.columns:
        df['store_size'] = df['store_size'].fillna('Unknown').astype(str)
    if {'store_location_tier', 'store_format'}.issubset(df.columns):
        df['store_tier_format'] = df['store_location_tier'].astype(str) + '_' + df['store_format'].astype(str)
    non_edible = ['household', 'health and hygiene', 'hard drinks', 'others']
    if 'product_category' in df.columns:
        df['fat_content_adjusted'] = np.where(df['product_category'].isin(non_edible), 'Non-Edible', df['fat_content'].fillna('Unknown'))
    return df

train = prepare_data(train)
test = prepare_data(test)
display(train.head())

In [ ]:
TARGET = 'total_sales'
ID_COL = 'id'
cat_features = [
    'product_code', 'fat_content', 'product_category', 'store_code',
    'store_size', 'store_location_tier', 'store_format', 'store_tier_format',
    'fat_content_adjusted'
]

for col in cat_features:
    if col in train.columns:
        train[col] = train[col].fillna('Unknown').astype(str)
    if col in test.columns:
        test[col] = test[col].fillna('Unknown').astype(str)

X = train.drop(columns=[ID_COL, TARGET])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])
cat_idx = [X.columns.get_loc(c) for c in cat_features if c in X.columns]
print('Categorical feature indexes:', cat_idx)
print('Feature count:', X.shape[1])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
    model = CatBoostRegressor(
        iterations=1000,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=3,
        random_strength=1,
        cat_features=cat_idx,
        random_seed=42,
        verbose=False,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=(X.iloc[va_idx], y.iloc[va_idx]), early_stopping_rounds=50)
    pred = np.clip(model.predict(X.iloc[va_idx]), 0, None)
    rmse = root_mean_squared_error(y.iloc[va_idx], pred)
    fold_scores.append(rmse)
    print(f'Fold {fold}: RMSE = {rmse:.4f}')

print(f'Mean CV RMSE: {np.mean(fold_scores):.4f}')
print(f'Std CV RMSE: {np.std(fold_scores):.4f}')

In [ ]:
final_model = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    random_strength=1,
    cat_features=cat_idx,
    random_seed=42,
    verbose=100,
)
final_model.fit(X, y)
test_predictions = np.clip(final_model.predict(X_test), 0, None)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_predictions})
submission.to_csv('final_catboost_submission.csv', index=False)
display(submission.head())
print('Submission saved to final_catboost_submission.csv')